In [28]:
import gym_electric_motor as gem
from gym_electric_motor.physical_systems import (
    SynchronousMotorSystem,
    IdealVoltageSupply,
    ContB6BridgeConverter,
    PermanentMagnetSynchronousMotor,
    MechanicalLoad
)
import numpy as np

# ===== Nguồn DC =====
supply = IdealVoltageSupply(u_nominal=48)

# ===== Bộ nghịch lưu B6 =====
converter = ContB6BridgeConverter()

# ===== PMSM =====
motor = PermanentMagnetSynchronousMotor(
    motor_parameter=dict(
        p=2,              # số đôi cực
        r_s=0.05,         # điện trở stator (Ω)
        l_d=0.0001,       # cảm kháng d-axis (H)
        l_q=0.0001,       # cảm kháng q-axis (H)
        psi_p=0.015,      # từ thông rotor (Wb)
        j_rotor=0.0001    # momen quán tính rotor (kg·m²)
    ),
    nominal_values=dict(
        omega=400,        # rad/s
        torque=2          # Nm
    )
)

# ===== Tải cơ học =====
# Phiên bản mới: vẫn dùng load_type và load_parameter
load = MechanicalLoad(load_type='Const', load_parameter=0.1)  # tải hằng số 0.1 Nm

# ===== Hệ thống PMSM =====
phys_sys = SynchronousMotorSystem(
    supply=supply,
    converter=converter,
    motor=motor,
    load=load,
    tau=1e-4
)

# ===== Tạo môi trường =====
env = gem.core.ContEnv(
    physical_system=phys_sys,
    reference_generator=None,
    reward_function=None
)
base_env = env.unwrapped

# ===== Lấy index và giới hạn tốc độ =====
omega_index = base_env.state_names.index('omega')
omega_limit = base_env.limits[omega_index]

# ===== Reset env =====
state, info = env.reset()

# ===== Chạy mô phỏng =====
for t in range(200):
    # duty = 0.2 trước step 100, 0.8 sau step 100
    duty = 0.2 if t < 100 else 0.8
    
    # Scale về [-1,1] cho ContB6BridgeConverter
    duty_scaled = 2*duty - 1
    action = np.array([duty_scaled]*3)
    
    (state_values, _), _, _, _, _ = env.step(action)
    
    # Tính RPM
    omega_mech = state_values[omega_index] * omega_limit
    rpm = omega_mech * 60 / (2*np.pi)
    
    print(f"Step {t:03d} | Duty={duty:.2f} | RPM={rpm:.1f}")


TypeError: MechanicalLoad.__init__() got an unexpected keyword argument 'load_type'

In [9]:
print(base_env.state_names)


['omega', 'torque', 'i_a', 'i_b', 'i_c', 'i_sd', 'i_sq', 'u_a', 'u_b', 'u_c', 'u_sd', 'u_sq', 'epsilon', 'u_sup']
